In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as f
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4

mps


-------  
**Why is it called bigram language model?**  
When we have a character we predict the second one, thus focusing on two characters at a time. Each prediction only depends on the one character right before it  that pairing (character, next character) is the "bigram."  

In [10]:
with open('sherlock_holmes.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)
print(vocab_size)

['\n', ' ', '!', '&', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '£', '½', 'à', 'â', 'æ', 'è', 'é', 'œ', '—', '‘', '’', '“', '”']
88


----  
**Tokenizer -> Encoder + Decoder**  
Encoder converts each element of the array into an integer  
*we're using a character level tokenizer here.*

In [11]:
#generator loops
string_to_int = {ch : i for i, ch in enumerate(chars)} #string to integer mapping
int_to_string = {i : ch for i, ch in enumerate(chars)} #integer to string mapping
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([41, 56, 53,  1, 22, 52, 70, 53, 62, 68, 69, 66, 53, 67,  1, 63, 54,  1,
        40, 56, 53, 66, 60, 63, 51, 59,  1, 29, 63, 60, 61, 53, 67,  0,  0, 50,
        73,  1, 22, 66, 68, 56, 69, 66,  1, 24, 63, 62, 49, 62,  1, 25, 63, 73,
        60, 53,  0,  0,  0, 24, 63, 62, 68, 53, 62, 68, 67,  0,  0,  1,  1,  1,
        30,  8,  1,  1,  1,  1,  1, 22,  1, 40, 51, 49, 62, 52, 49, 60,  1, 57,
        62,  1, 23, 63, 56, 53, 61, 57, 49,  0])


In [12]:
#training and validation split
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y
x, y = get_batch('train')
print('inputs: ')
# print x.shape
print(x)
print('targets: ')
print(y)

tensor([ 47261, 174875,  77541, 375923])
inputs: 
tensor([[66, 60, 73,  1, 55, 53, 62, 68],
        [71, 56, 53, 66, 53,  1, 67, 68],
        [ 1, 59, 53, 53, 62,  7, 71, 57],
        [56, 49, 68,  1, 57, 68,  1, 57]], device='mps:0')
targets: 
tensor([[60, 73,  1, 55, 53, 62, 68, 60],
        [56, 53, 66, 53,  1, 67, 68, 66],
        [59, 53, 53, 62,  7, 71, 57, 68],
        [49, 68,  1, 57, 68,  1, 57, 67]], device='mps:0')


In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

#sequential -> CPU
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([41]) target is tensor(56)
when input is tensor([41, 56]) target is tensor(53)
when input is tensor([41, 56, 53]) target is tensor(1)
when input is tensor([41, 56, 53,  1]) target is tensor(22)
when input is tensor([41, 56, 53,  1, 22]) target is tensor(52)
when input is tensor([41, 56, 53,  1, 22, 52]) target is tensor(70)
when input is tensor([41, 56, 53,  1, 22, 52, 70]) target is tensor(53)
when input is tensor([41, 56, 53,  1, 22, 52, 70, 53]) target is tensor(62)


---  
Block size - size of sequence  
Batch size - how many sequences/blocks we can process parallely  
*without these you can't expect fast training and good performance.*

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocan_size):
        super(.__init__()
        # gonna be a learnable para because of the nn
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

        # forward pass function helps us understand the inderlying ops of the model.
    def forward(self, index, targets):
        logits = self.token_embedding_table(index)
        return logits